In [14]:
# Cell 1 — Imports
import os, time, re, json, math
from typing import Dict, Any, List, Tuple, Optional

import requests
import pandas as pd
from pandas.tseries.offsets import MonthEnd


In [15]:
# Cell 2 — Config

# >>> Put your IndianAPI key here <<<
INDIANAPI_KEY = os.getenv("INDIANAPI_KEY", "sk-live-qRes50UXfu48WwWyW8FEavml5F6GP7kymfOeC9Wb")  # replace if not using env

# Base + I/O
INDIANAPI_BASE = "https://stock.indianapi.in"
INPUT_XLSX     = "India_Top_45_Companies_By_Sector.xlsx"   # your 45 companies list
OUTPUT_XLSX    = "raw_indianapi_quarterly.xlsx"
NO_DATA_CSV    = "no_data_companies.csv"

# Column names expected in the 45 list
COL_COMPANY = "Company"
COL_TICKER  = "Ticker"            # IndianAPI stock_name
COL_SECTOR  = "Sector"
COL_CAP     = "Cap Category"

# Minimal filters for board-meeting agendas we treat as "results" announcements
RESULT_KEYWORDS = (
    "Quarterly Results",
    "Financial Results",
    "Results for the quarter",
    "approval of financial results",
    "results for the quarter",
    "audited condensed standalone",
    "audited consolidated financial results",
)

# Requests / retry
REQ_TIMEOUT_S = 30
MAX_RETRIES   = 3
RETRY_BACKOFF = 1.8  # seconds multiplier


In [16]:
# Cell 3 — Load companies (45)

def load_companies(xlsx_path: str) -> pd.DataFrame:
    df = pd.read_excel(xlsx_path)
    # Keep only the columns we need (ignore extra Unnamed cols from Excel)
    keep = [c for c in df.columns if c in {COL_COMPANY, COL_TICKER, COL_SECTOR, COL_CAP}]
    df = df[keep].copy()
    # Basic cleanup
    df[COL_COMPANY] = df[COL_COMPANY].astype(str).str.strip()
    df[COL_TICKER]  = df[COL_TICKER].astype(str).str.strip().str.upper()
    if COL_SECTOR in df:
        df[COL_SECTOR] = df[COL_SECTOR].astype(str).str.strip()
    if COL_CAP in df:
        df[COL_CAP] = df[COL_CAP].astype(str).str.strip()
    # Drop rows missing ticker/company
    df = df[(df[COL_COMPANY] != "") & (df[COL_TICKER] != "")]
    return df

companies = load_companies(INPUT_XLSX)
print(f"Loaded companies: {len(companies)}")
companies.head(3)


Loaded companies: 45


,Company,Ticker,Sector,Cap Category
0,Tata Consultancy Services,TCS,IT / Technology,Large
1,Infosys,INFY,IT / Technology,Large
2,HCL Technologies,HCLTECH,IT / Technology,Large


In [21]:
# Cell 4 — IndianAPI HTTP helpers (hardened)

import requests, time, json
from typing import Dict, Any, Optional, Tuple, List

# IMPORTANT: API_KEY must be defined in Cell 1 as `API_KEY = "<your key>"`
assert isinstance(API_KEY, str) and API_KEY.strip(), "API_KEY missing in Cell 1"

# Some accounts see the same routes at either host; try both.
_INDIANAPI_HOSTS = [
    "https://stock.indianapi.in"
    
]

# One session for connection reuse
_SESSION = requests.Session()
_COMMON_HEADERS = {
    "x-api-key": API_KEY,
    # A real UA reduces chance of being rejected by WAFs/CDNs.
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json",
}

def _explain_failure(r: requests.Response) -> str:
    """Return a short string with status + first chars of body for debugging."""
    snippet = ""
    try:
        # Try JSON first
        j = r.json()
        snippet = json.dumps(j, ensure_ascii=False)[:400]
    except Exception:
        # Fallback to text
        snippet = (r.text or "")[:400]
    return f"{r.status_code} for {r.url} | body: {snippet!r}"

def _get_json(
    path: str,
    params: Optional[Dict[str, Any]] = None,
    method: str = "GET",
    retries: int = 2,
    timeout: Tuple[float, float] = (10, 30),
) -> Any:
    """
    Calls IndianAPI with headers and returns parsed JSON.
    Tries both known hosts; on 4xx/5xx logs a helpful snippet before raising.
    """
    errors: List[str] = []
    params = params or {}

    for host in _INDIANAPI_HOSTS:
        url = f"{host.rstrip('/')}/{path.lstrip('/')}"
        for attempt in range(retries + 1):
            try:
                if method.upper() == "GET":
                    r = _SESSION.get(url, params=params, headers=_COMMON_HEADERS,
                                     timeout=timeout, allow_redirects=True)
                else:
                    r = _SESSION.post(url, json=params, headers=_COMMON_HEADERS,
                                      timeout=timeout, allow_redirects=True)

                if r.status_code >= 400:
                    # Collect helpful details (server error body often tells you exactly what's wrong)
                    errors.append(_explain_failure(r))
                    # Short backoff on transient stuff
                    if 500 <= r.status_code < 600 and attempt < retries:
                        time.sleep(1.2 * (attempt + 1))
                        continue
                    # For 4xx, no point retrying the same host unless backoff helps WAF; still continue to next host
                    break

                # OK
                try:
                    return r.json()
                except Exception as e:
                    errors.append(f"JSON decode error on {r.url}: {e}")
                    break

            except requests.RequestException as e:
                errors.append(f"{type(e).__name__} on {url}: {e}")
                if attempt < retries:
                    time.sleep(1.2 * (attempt + 1))
                    continue
                break

    raise requests.HTTPError("IndianAPI request failed. Attempts:\n  - " + "\n  - ".join(errors))


In [22]:
# Cell 5 — IndianAPI fetchers

def fetch_quarter_results(stock_name: str) -> Dict[str, Dict[str, Any]]:
    """
    /historical_stats?stock_name={}&stats=quarter_results
    Returns dict like: {"Sales": {"Jun 2025": 63437, ...}, "OPM %": {...}, ...}
    """
    params = {"stock_name": stock_name, "stats": "quarter_results"}
    raw = _get_json("historical_stats", params=params)

    if not isinstance(raw, dict) or not raw:
        raise RuntimeError(f"quarter_results empty/invalid for {stock_name}")

    # Some sandboxes wrap under 'datasets' etc for other endpoints; for quarterly it's dict of metrics.
    # sanity: at least one metric should be a dict with string keys like 'Jun 2025'
    good = False
    for k, v in raw.items():
        if isinstance(v, dict) and v:
            good = True
            break
    if not good:
        raise RuntimeError(f"quarter_results structure unexpected for {stock_name}")
    return raw

def fetch_corporate_actions(stock_name: str) -> Dict[str, Any]:
    """
    /corporate_actions?stock_name={}
    We use 'board_meetings'->'data' to find “Quarterly Results” dates.
    """
    params = {"stock_name": stock_name}
    raw = _get_json("corporate_actions", params=params)
    if not isinstance(raw, dict) or "board_meetings" not in raw:
        # Some stocks might not have board_meetings
        return {"board_meetings": {"header": ["Date", "Agenda"], "data": []}}
    return raw


In [23]:
# Cell 6 — Parsing helpers (lightweight, no heavy normalization)

_date_pat = re.compile(r"^\s*(\d{1,2})[-/](\d{1,2})[-/](\d{4})\s*$")

def parse_india_date(s: str) -> Optional[pd.Timestamp]:
    """Parse dates like '10-07-2025' or '10/07/2025' -> Timestamp('2025-07-10')"""
    if not isinstance(s, str):
        return None
    m = _date_pat.match(s.strip())
    if not m:
        return None
    d, mth, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
    try:
        return pd.Timestamp(year=y, month=mth, day=d)
    except Exception:
        return None

def board_meeting_result_dates(board_json: Dict[str, Any]) -> List[pd.Timestamp]:
    """Extract dates from board_meetings where Agenda hints results."""
    out: List[pd.Timestamp] = []
    bm = board_json.get("board_meetings", {})
    data = bm.get("data", [])
    for row in data:
        if not isinstance(row, (list, tuple)) or len(row) < 2:
            continue
        d_raw, agenda = str(row[0]), str(row[1])
        d = parse_india_date(d_raw)
        if d is None:
            continue
        text = agenda.lower()
        if any(key.lower() in text for key in RESULT_KEYWORDS):
            out.append(d.normalize())
    # distinct + sort
    out = sorted(pd.Series(out).drop_duplicates().tolist())
    return out

def month_end_from_label(period_label: str) -> Optional[pd.Timestamp]:
    """
    'Jun 2025' -> 2025-06-30, 'Sep 2024' -> 2024-09-30, etc.
    """
    try:
        ts = pd.to_datetime("01 " + period_label)
        return (ts + MonthEnd(0)).normalize()
    except Exception:
        return None

def map_period_to_result_date(period_label: str, candidates: List[pd.Timestamp]) -> Optional[pd.Timestamp]:
    """
    Minimal alignment: for a quarter period (Mar/Jun/Sep/Dec), look for the first result date
    within 0..45 days after period end. If none, leave blank.
    """
    qend = month_end_from_label(period_label)
    if qend is None or not candidates:
        return None
    lo = qend
    hi = qend + pd.Timedelta(days=45)
    for d in candidates:
        if lo <= d <= hi:
            return d
    return None


In [24]:
# Cell 7 — Per-company assembly (test on one; used by the batch loop next)

RAW_ID_COLS = [COL_COMPANY, COL_TICKER, COL_SECTOR, COL_CAP]

def build_company_raw(stock_row: pd.Series) -> Tuple[pd.DataFrame, Optional[str]]:
    """
    Returns (df, error_msg). df has columns:
      Company, Ticker, Sector, Cap Category, Period, Result Date, <raw metric columns as-is from IndianAPI>
    No normalization/engineering on the metrics; we only try to attach Result Date (if found).
    """
    company = stock_row.get(COL_COMPANY)
    ticker  = stock_row.get(COL_TICKER)
    sector  = stock_row.get(COL_SECTOR, None)
    cap     = stock_row.get(COL_CAP, None)

    # fetch
    try:
        qraw = fetch_quarter_results(ticker)
    except Exception as e:
        return pd.DataFrame(), f"quarter_results failed: {e}"

    try:
        cacts = fetch_corporate_actions(ticker)
    except Exception as e:
        # not fatal
        cacts = {"board_meetings": {"header": ["Date", "Agenda"], "data": []}}

    # tidy quarterly without renaming metrics
    # qraw: metric -> {period: value}
    # First make a union of all periods
    all_periods: List[str] = []
    for metric, series in qraw.items():
        if isinstance(series, dict):
            all_periods.extend(list(series.keys()))
    periods = sorted(pd.Series(all_periods).dropna().drop_duplicates().tolist(),
                     key=lambda x: pd.to_datetime("01 " + str(x), errors="coerce") or pd.Timestamp.max)

    # Create frame
    df = pd.DataFrame({"Period": periods})
    # Attach each metric column as-is
    for metric, series in qraw.items():
        if not isinstance(series, dict):
            continue
        ser = pd.Series(series, name=metric)
        # Align by Period key
        df = df.merge(ser.rename_axis("Period").reset_index(),
                      on="Period", how="left")

    # Result dates (minimal alignment)
    rd_candidates = board_meeting_result_dates(cacts)
    df["Result Date"] = [
        map_period_to_result_date(p, rd_candidates) for p in df["Period"]
    ]
    # Meta
    df.insert(0, COL_CAP, cap)
    df.insert(0, COL_SECTOR, sector)
    df.insert(0, COL_TICKER, ticker)
    df.insert(0, COL_COMPANY, company)

    # keep order: meta, Period, Result Date, then raw metrics (as-is)
    return df, None

# --- quick test on TCS (assuming your sheet has the row) ---
tcs_row = companies[companies[COL_TICKER].str.upper() == "TCS"].iloc[0]
tcs_df, tcs_err = build_company_raw(tcs_row)
print("TCS error:", tcs_err)
display(tcs_df.head(8))


TCS error: None


,Company,Ticker,Sector,Cap Category,Period,Sales,Expenses,Operating Profit,OPM %,Other Income,Interest,Depreciation,Profit before tax,Tax %,Net Profit,EPS in Rs,Result Date
0,Tata Consultancy Services,TCS,IT / Technology,Large,Jun 2022,52758.0,39342.0,13416.0,25.0,789.0,199.0,1230.0,12776.0,25.0,9519.0,25.90,2022-07-08
1,Tata Consultancy Services,TCS,IT / Technology,Large,Sep 2022,55309.0,40793.0,14516.0,26.0,965.0,148.0,1237.0,14096.0,26.0,10465.0,28.51,2022-10-10
2,Tata Consultancy Services,TCS,IT / Technology,Large,Dec 2022,58229.0,42676.0,15553.0,27.0,520.0,160.0,1269.0,14644.0,26.0,10883.0,29.64,2023-01-09
3,Tata Consultancy Services,TCS,IT / Technology,Large,Mar 2023,59162.0,43388.0,15774.0,27.0,1175.0,272.0,1286.0,15391.0,26.0,11436.0,31.13,2023-04-12
4,Tata Consultancy Services,TCS,IT / Technology,Large,Jun 2023,59381.0,44383.0,14998.0,25.0,1397.0,163.0,1243.0,14989.0,26.0,11120.0,30.26,2023-07-12
5,Tata Consultancy Services,TCS,IT / Technology,Large,Sep 2023,59692.0,43946.0,15746.0,26.0,1006.0,159.0,1263.0,15330.0,26.0,11380.0,31.00,2023-10-11
6,Tata Consultancy Services,TCS,IT / Technology,Large,Dec 2023,60583.0,44195.0,16388.0,27.0,-96.0,230.0,1233.0,14829.0,25.0,11097.0,30.56,2024-01-11
7,Tata Consultancy Services,TCS,IT / Technology,Large,Mar 2024,61237.0,44073.0,17164.0,28.0,1157.0,226.0,1246.0,16849.0,26.0,12502.0,34.37,2024-04-12


In [25]:
# Cell 8 — Batch run for all 45; write files

all_rows: List[pd.DataFrame] = []
fail_rows: List[Dict[str, Any]] = []

for i, row in companies.reset_index(drop=True).iterrows():
    name  = row.get(COL_COMPANY)
    tick  = row.get(COL_TICKER)
    print(f"[{i+1}/{len(companies)}] {name} | IndianAPI={tick} ...", end=" ")
    try:
        df_i, err = build_company_raw(row)
        if err:
            fail_rows.append({"Company": name, "Ticker": tick, "Error": err})
            print("FAIL")
            continue
        if len(df_i) == 0 or df_i.drop(columns=[COL_COMPANY, COL_TICKER, COL_SECTOR, COL_CAP, "Period", "Result Date"], errors="ignore").isna().all(None):
            fail_rows.append({"Company": name, "Ticker": tick, "Error": "empty quarterly payload"})
            print("EMPTY")
            continue
        all_rows.append(df_i)
        print(f"OK ({len(df_i)} rows)")
        # polite pause to avoid rate limits
        time.sleep(0.2)
    except Exception as e:
        fail_rows.append({"Company": name, "Ticker": tick, "Error": str(e)})
        print("ERROR")

# Save success rows
if all_rows:
    raw_all = pd.concat(all_rows, ignore_index=True)
    with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as xw:
        raw_all.to_excel(xw, sheet_name="raw", index=False)
    print(f"\nSaved RAW rows -> {OUTPUT_XLSX} ({len(raw_all)} rows)")
else:
    print("\nNo raw rows to save.")

# Save failures list
no_data_df = pd.DataFrame(fail_rows) if fail_rows else pd.DataFrame(columns=["Company", "Ticker", "Error"])
no_data_df.to_csv(NO_DATA_CSV, index=False)
print(f"Saved NO-DATA list -> {NO_DATA_CSV} ({len(no_data_df)} companies)")
display(no_data_df)


[1/45] Tata Consultancy Services | IndianAPI=TCS ... OK (13 rows)
[2/45] Infosys | IndianAPI=INFY ... OK (13 rows)
[3/45] HCL Technologies | IndianAPI=HCLTECH ... OK (13 rows)
[4/45] Tech Mahindra | IndianAPI=TECHM ... OK (13 rows)
[5/45] LTIMindtree | IndianAPI=LTIM ... OK (13 rows)
[6/45] Wipro | IndianAPI=WIPRO ... OK (13 rows)
[7/45] Coforge | IndianAPI=COFORGE ... OK (13 rows)
[8/45] Persistent Systems | IndianAPI=PERSISTENT ... OK (13 rows)
[9/45] Zensar Tech | IndianAPI=ZENSARTECH ... OK (13 rows)
[10/45] HDFC Bank | IndianAPI=HDFCBANK ... OK (13 rows)
[11/45] ICICI Bank | IndianAPI=ICICIBANK ... OK (13 rows)
[12/45] SBI | IndianAPI=SBIN ... OK (13 rows)
[13/45] Axis Bank | IndianAPI=AXISBANK ... OK (13 rows)
[14/45] Kotak Mahindra Bank | IndianAPI=KOTAKBANK ... OK (13 rows)
[15/45] Bajaj Finance | IndianAPI=BAJFINANCE ... OK (13 rows)
[16/45] IDFC First Bank | IndianAPI=IDFCFIRSTB ... OK (13 rows)
[17/45] AU Small Finance Bank | IndianAPI=AUBANK ... OK (13 rows)
[18/45] LIC Hou

,Company,Ticker,Error
0,Bajaj Auto,BAJAJ-AUTO,quarter_results failed: quarter_results struct...


In [31]:
# One-cell runner — fetch IndianAPI data for ONE company (quarter_results + board_meetings)
# ------------------------------------------------------------------------------

# >>>> Put your IndianAPI key here (or make sure API_KEY already exists in the kernel) <<<<
API_KEY = globals().get("API_KEY", "").strip() or "YOUR_INDIANAPI_KEY_HERE"

# Company you want to test:
COMPANY_NAME = "Bajaj Auto"
STOCK_NAME   = "Bajaj Auto"   # IndianAPI stock_name, e.g. TCS, INFY, HDFCBANK, ITC, etc.

# ------------------------------------------------------------------------------
# Everything below is self-contained (HTTP helpers + fetchers + run)
# ------------------------------------------------------------------------------

import json, time, requests, pandas as pd
from typing import Any, Dict, Optional, Tuple, List
from IPython.display import display

assert API_KEY and API_KEY != "YOUR_INDIANAPI_KEY_HERE", "Please set API_KEY with your IndianAPI key."

# Try both hosts; some tenants route to 'analyst' instead of 'stock'
_INDIANAPI_HOSTS = [
    "https://stock.indianapi.in"   
]

_SESSION = requests.Session()
_COMMON_HEADERS = {
    "x-api-key": API_KEY,
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json",
}

def _err_snippet(r: requests.Response) -> str:
    try:
        return json.dumps(r.json(), ensure_ascii=False)[:400]
    except Exception:
        return (r.text or "")[:400]

def _get_json(
    path: str,
    params: Optional[Dict[str, Any]] = None,
    method: str = "GET",
    retries: int = 1,
    timeout: Tuple[float, float] = (10, 30),
) -> Any:
    """Hardened GET/POST to IndianAPI with helpful error text and dual-host fallback."""
    params = params or {}
    errors: List[str] = []
    for host in _INDIANAPI_HOSTS:
        url = f"{host.rstrip('/')}/{path.lstrip('/')}"
        for attempt in range(retries + 1):
            try:
                if method.upper() == "GET":
                    r = _SESSION.get(url, params=params, headers=_COMMON_HEADERS, timeout=timeout)
                else:
                    r = _SESSION.post(url, json=params, headers=_COMMON_HEADERS, timeout=timeout)

                if r.status_code >= 400:
                    errors.append(f"{r.status_code} {r.reason} for {r.url} | body: {_err_snippet(r)!r}")
                    if 500 <= r.status_code < 600 and attempt < retries:
                        time.sleep(1.2 * (attempt + 1))
                        continue
                    break  # on 4xx or after retries on 5xx
                try:
                    return r.json()
                except Exception as e:
                    errors.append(f"JSON decode error on {r.url}: {e}")
                    break
            except requests.RequestException as e:
                errors.append(f"{type(e).__name__} on {url}: {e}")
                if attempt < retries:
                    time.sleep(1.2 * (attempt + 1))
                    continue
                break
    raise requests.HTTPError("IndianAPI request failed:\n  - " + "\n  - ".join(errors))

def fetch_quarter_results_raw(stock_name: str) -> pd.DataFrame:
    """
    GET /historical_stats?stock_name=<ticker>&stats=quarter_results
    Returns the RAW frame (no normalization/engineering).
    """
    params = {"stock_name": stock_name, "stats": "quarter_results"}
    try:
        raw = _get_json("historical_stats", params, method="GET")
    except requests.HTTPError:
        # Some tenants accept POST for this route — try it once.
        raw = _get_json("historical_stats", params, method="POST")

    if not isinstance(raw, dict) or not raw:
        raise RuntimeError(f"Unexpected payload for quarter_results of {stock_name}: {type(raw)}")
    # raw is a dict: metric -> { 'Jun 2022': value, ... }
    df = pd.DataFrame(raw).T  # metrics as rows
    df = df.T.reset_index().rename(columns={"index": "Period"})  # Period as a column (e.g., 'Jun 2025')
    return df  # RAW

def fetch_board_meetings_raw(stock_name: str) -> pd.DataFrame:
    """
    GET /corporate_actions?stock_name=<ticker>
    Extract raw board_meetings table (Date, Agenda) — keep raw.
    """
    payload = _get_json("corporate_actions", {"stock_name": stock_name}, method="GET")

    if not isinstance(payload, dict) or "board_meetings" not in payload:
        raise RuntimeError(f"Unexpected corporate_actions payload for {stock_name}. Keys: {list(payload)[:8] if isinstance(payload, dict) else type(payload)}")

    bm = payload["board_meetings"]
    header = bm.get("header", [])
    data   = bm.get("data", [])
    if not data:
        return pd.DataFrame(columns=["Date", "Agenda"])

    df = pd.DataFrame(data, columns=header if header else ["Date", "Agenda"])
    # Keep RAW, only parse Date to datetime safely (dayfirst True matches Indian format)
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    return df

# ----------------------------
# Run for ONE company
# ----------------------------
print(f"=== Fetching for: {COMPANY_NAME} | IndianAPI={STOCK_NAME} ===")

# Quarterly results (RAW)
try:
    q_df = fetch_quarter_results_raw(STOCK_NAME)
    print(f"[quarter_results] rows={len(q_df)} cols={list(q_df.columns)}")
    display(q_df.tail(10))
    q_path = f"{STOCK_NAME}_quarter_results_raw.csv"
    q_df.to_csv(q_path, index=False)
    print(f"Saved quarterly raw -> {q_path}")
except Exception as e:
    print(f"quarter_results FAILED for {STOCK_NAME}: {e}")
    q_df = pd.DataFrame()

# Board meetings (RAW)
try:
    bm_df = fetch_board_meetings_raw(STOCK_NAME)
    #bm_df = fetch_board_meetings_raw("Bajaj Auto")
    print(f"[board_meetings] rows={len(bm_df)} cols={list(bm_df.columns)}")
    display(bm_df.sort_values("Date").tail(10))
    bm_path = f"{STOCK_NAME}_board_meetings_raw.csv"
    bm_df.to_csv(bm_path, index=False)
    print(f"Saved board_meetings raw -> {bm_path}")
except Exception as e:
    print(f"board_meetings FAILED for {STOCK_NAME}: {e}")
    bm_df = pd.DataFrame()

# Return objects to the notebook namespace for your further use
q_df, bm_df


=== Fetching for: Bajaj Auto | IndianAPI=Bajaj Auto ===
[quarter_results] rows=13 cols=['Period', 'Sales', 'Expenses', 'Operating Profit', 'OPM %', 'Other Income', 'Interest', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit', 'EPS in Rs']


,Period,Sales,Expenses,Operating Profit,OPM %,Other Income,Interest,Depreciation,Profit before tax,Tax %,Net Profit,EPS in Rs
3,Mar 2023,8929.0,7272.0,1657.0,19.0,595.0,16.0,76.0,2160.0,21.0,1705.0,60.25
4,Jun 2023,10312.0,8380.0,1932.0,19.0,351.0,12.0,87.0,2184.0,25.0,1644.0,58.11
5,Sep 2023,10838.0,8708.0,2130.0,20.0,552.0,7.0,92.0,2584.0,22.0,2020.0,71.39
6,Dec 2023,12165.0,9750.0,2415.0,20.0,356.0,12.0,93.0,2666.0,24.0,2033.0,71.78
7,Mar 2024,11555.0,9271.0,2284.0,20.0,444.0,30.0,93.0,2606.0,23.0,2011.0,72.05
8,Jun 2024,11932.0,9562.0,2370.0,20.0,335.0,47.0,95.0,2564.0,24.0,1942.0,69.55
9,Sep 2024,13247.0,11174.0,2073.0,16.0,399.0,75.0,98.0,2299.0,40.0,1385.0,49.61
10,Dec 2024,13169.0,10418.0,2751.0,21.0,348.0,120.0,102.0,2876.0,24.0,2196.0,78.62
11,Mar 2025,12646.0,10289.0,2358.0,19.0,392.0,147.0,119.0,2484.0,27.0,1802.0,64.52
12,Jun 2025,13133.0,10340.0,2793.0,21.0,509.0,224.0,118.0,2961.0,25.0,2210.0,79.15


Saved quarterly raw -> Bajaj Auto_quarter_results_raw.csv
[board_meetings] rows=92 cols=['Date', 'Agenda']


,Date,Agenda
10,2024-07-16,BAJAJ AUTO LTD.has informed BSE that the meeti...
8,2024-10-16,BAJAJ AUTO LTD.has informed BSE that the meeti...
7,2025-01-28,BAJAJ AUTO LTD.has informed BSE that the meeti...
6,2025-03-18,Outcome of Board Meeting - Re-appointment of M...
3,2025-05-15,Additional capital infusion by Bajaj Auto Limi...
2,2025-05-22,Intimation of agreements entered into by Bajaj...
5,2025-05-29,Bajaj Auto Ltdhas informed BSE that the meetin...
4,2025-05-29,Quarterly Results
1,2025-08-06,Quarterly Results Outcome of the Board Meeting...
0,2025-08-06,Quarterly Results


Saved board_meetings raw -> Bajaj Auto_board_meetings_raw.csv


(      Period    Sales  Expenses  Operating Profit  OPM %  Other Income  Interest  Depreciation  Profit before tax  Tax %  Net Profit  EPS in Rs
 0   Jun 2022   8005.0    6719.0            1286.0   16.0         320.0       4.0          68.0             1534.0   24.0      1163.0      40.20
 1   Sep 2022  10203.0    8453.0            1750.0   17.0         532.0      11.0          67.0             2203.0   22.0      1719.0      59.42
 2   Dec 2022   9319.0    7561.0            1757.0   19.0         271.0       8.0          75.0             1945.0   24.0      1473.0      52.05
 3   Mar 2023   8929.0    7272.0            1657.0   19.0         595.0      16.0          76.0             2160.0   21.0      1705.0      60.25
 4   Jun 2023  10312.0    8380.0            1932.0   19.0         351.0      12.0          87.0             2184.0   25.0      1644.0      58.11
 5   Sep 2023  10838.0    8708.0            2130.0   20.0         552.0       7.0          92.0             2584.0   22.0      202